# 02 — Training and Evaluation

This notebook is the second part of a minimal working example demonstrating a
**coupled sampling–training framework** for MLIP development.

Run this notebook **after** VASP DFT calculations are complete and results
have been collected into ExtXYZ format.

The key idea is that sampling and training are tightly coupled:
the three sampling strategies from notebook 01 each produce a different
augmented dataset, which in turn produces a different fine-tuned model.
By evaluating all models on the same test set you can directly compare
how Random, DIRECT, and LCMD sampling affect model accuracy.

1. **Scratch training** — SevenNet trained on the 400-structure foundational dataset  
2. **Fine-tuning** — one fine-tuned model per sampling method (400 + 100 structures)  
3. **Evaluation** — energy / force parity plots and error metrics  
4. **Analysis** — cluster statistics, force RMSE by distance bin, violin plots

---
### Acknowledgements

This pipeline uses
[maml](https://github.com/materialsvirtuallab/maml) for M3GNet feature extraction and
[SevenNet](https://github.com/MDIL-SNU/SevenNet) for MLIP training.
Code was written with the assistance of [Claude](https://claude.ai) (Anthropic).

---
### Expected inputs
```
data/1_example_data/
    v1_dataset/
        initial_200fs_500.xyz
        pca_model/
            initial_200fs_500_reduced.h5
            augmented_10fs_10000_reduced.h5
    v1_models/
        sampling/
            random_selected.xyz   ← DFT-labelled (100 structures each)
            direct_selected.xyz
            lcmd_selected.xyz
            test_selected.xyz     ← held-out test set (DFT labelled)
```

In [ ]:
import os, sys, json
import numpy as np
sys.path.insert(0, os.path.abspath('.'))

from ase.io import read as ase_read, write as ase_write
from src.training import train_scratch, fine_tune, evaluate_model
from src.sampling import load_reduced_h5
from src.plotting import plot_parity, plot_cluster_analysis, plot_force_rmse_by_bin, plot_violin

# dataset and output root directories
DATASET_DIR  = 'data/1_example_data/v1_dataset'
MODELS_DIR   = 'data/1_example_data/v1_models'

# input files
CENTER_XYZ   = f'{DATASET_DIR}/initial_200fs_500.xyz'
CENTER_H5    = f'{DATASET_DIR}/pca_model/initial_200fs_500_reduced.h5'
CAND_H5      = f'{DATASET_DIR}/pca_model/augmented_10fs_10000_reduced.h5'

# directories for outputs from this notebook
SAMPLING_DIR = f'{MODELS_DIR}/sampling'
TRAINING_DIR = f'{MODELS_DIR}/training'
RESULTS_DIR  = f'{MODELS_DIR}/results'
TEST_XYZ     = f'{SAMPLING_DIR}/test_selected.xyz'

os.makedirs(TRAINING_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,  exist_ok=True)

## Step 1 — Scratch training

A SevenNet model is trained from random initialisation on the foundational
dataset only (400 structures). This gives a baseline model that has seen no
augmented data, and serves as the shared starting point for all fine-tuning
runs — ensuring any difference between the fine-tuned models is attributable
solely to the sampling strategy.

In [ ]:
# extract the first 400 structures as the foundational set
foundational = ase_read(CENTER_XYZ, index=':')
foundational = foundational[:400]
foundational_xyz = f'{RESULTS_DIR}/foundational_400.xyz'
ase_write(foundational_xyz, foundational)
print(f'{len(foundational)} foundational structures → {foundational_xyz}')

In [ ]:
# train SevenNet from scratch on the foundational set;
# this checkpoint is the base for all fine-tuning runs below
scratch_ckpt = train_scratch(
    train_xyz     = foundational_xyz,
    output_dir    = f'{TRAINING_DIR}/scratch',
    cutoff        = 5.0,
    total_epochs  = 50,
    batch_size    = 4,
    lr            = 0.01,
    force_weight  = 0.1,
    stress_weight = 0.01,
    device        = 'cuda',
)
print(f'Scratch checkpoint → {scratch_ckpt}')

## Step 2 — Fine-tuning

For each sampling method, the scratch checkpoint is fine-tuned on the union
of the 400 foundational structures and the 100 DFT-labelled sampled structures.
This produces three separate models whose differences stem entirely from which
100 structures were selected in notebook 01.

In [ ]:
ft_checkpoints = {}

for method in ['random', 'direct', 'lcmd']:
    xyz_path = f'{SAMPLING_DIR}/{method}_selected.xyz'

    # skip if DFT-labelled xyz not yet available
    if not os.path.exists(xyz_path):
        print(f'[SKIP] {method}: {xyz_path} not found — run VASP first')
        continue

    # fine-tune from scratch_ckpt on foundational (400) + sampled (100)
    ckpt = fine_tune(
        base_checkpoint = scratch_ckpt,
        augment_xyz     = xyz_path,
        center_xyz      = foundational_xyz,
        output_dir      = f'{TRAINING_DIR}/ft_{method}',
        total_epochs    = 10,
        batch_size      = 4,
        lr              = 0.004,
        force_weight    = 0.1,
        stress_weight   = 0.01,
        device          = 'cuda',
    )
    ft_checkpoints[method] = ckpt
    print(f'{method} fine-tuned checkpoint → {ckpt}')

## Step 3 — Evaluation on test set

In [ ]:
eval_results = {}

# evaluate the scratch model — foundational dataset only, no augmentation
eval_results['scratch'] = evaluate_model(
    checkpoint = scratch_ckpt,
    test_xyz   = TEST_XYZ,
    device     = 'cuda',
)

# evaluate each fine-tuned model on the same test set for fair comparison
for method, ckpt in ft_checkpoints.items():
    eval_results[f'ft_{method}'] = evaluate_model(
        checkpoint = ckpt,
        test_xyz   = TEST_XYZ,
        device     = 'cuda',
    )

# print summary table
header = f'{"Model":<18} | {"E-RMSE":>10} | {"E-MAE":>10} | {"F-RMSE":>10} | {"F-MAE":>10}'
print()
print(header)
print('-' * len(header))
for label, r in eval_results.items():
    print(f'{label:<18} | {r["energy_rmse"]:10.4f} | {r["energy_mae"]:10.4f} '
          f'| {r["force_rmse"]:10.4f} | {r["force_mae"]:10.4f}')

In [ ]:
# density-coloured parity plot for energy and force across all models
plot_parity(
    results     = eval_results,
    output_path = f'{RESULTS_DIR}/parity_all.png',
    title       = 'Energy & Force Parity',
)

## Step 4 — Analysis

### 4a. Cluster analysis

Birch cluster statistics are computed at 10 equal increments (10 → 100)
for each sampling method. The plots show how mean cluster size, maximum
cluster size, and coefficient of variation evolve as more structures are
added — a proxy for how well each method covers the feature space.

In [ ]:
from sklearn.cluster import Birch

cand_feats, _       = load_reduced_h5(CAND_H5)
center_feats_all, _ = load_reduced_h5(CENTER_H5)
center_feats        = center_feats_all[:400]

def cluster_stats(feats, threshold=1.0):
    # fit Birch and return statistics on non-singleton cluster sizes
    labels = Birch(n_clusters=None, threshold=threshold).fit_predict(feats)
    _, counts = np.unique(labels, return_counts=True)
    active = counts[counts > 1]
    if len(active) == 0:
        return {'mean': 0.0, 'std': 0.0, 'max': 0.0}
    return {'mean': float(active.mean()), 'std': float(active.std()), 'max': float(active.max())}

# compute stats at 10 equal increments up to 100 selected structures
increments   = list(range(10, 101, 10))
cluster_data = {}

for method in ['random', 'direct', 'lcmd']:
    json_path = f'{SAMPLING_DIR}/{method}_selected.json'
    if not os.path.exists(json_path):
        continue
    with open(json_path) as f:
        all_idx = [s['original_index'] for s in json.load(f)['structures']]

    sizes, means, stds, maxs = [], [], [], []
    for n in increments:
        s = cluster_stats(cand_feats[all_idx[:n]])
        sizes.append(n); means.append(s['mean']); stds.append(s['std']); maxs.append(s['max'])
    cluster_data[method] = {'sizes': sizes, 'mean': means, 'std': stds, 'max': maxs}

# foundational set used as a reference baseline in the plot
ref_stats = cluster_stats(center_feats)

plot_cluster_analysis(
    data        = cluster_data,
    output_path = f'{RESULTS_DIR}/cluster_analysis.png',
    reference   = ref_stats,
)

### 4b. Force RMSE binned by nearest-neighbour distance

Per-atom force errors are grouped by each atom's nearest-neighbour distance.
This reveals whether errors are concentrated at short (compressed) or long
(under-sampled) interatomic separations.

In [ ]:
import gc, torch
from ase.neighborlist import neighbor_list
from sevenn.calculator import SevenNetCalculator

def force_rmse_by_distance(checkpoint, test_xyz, cutoff=5.0, bin_width=1.0):
    traj = ase_read(test_xyz, index=':')
    calc = SevenNetCalculator(checkpoint)
    bin_errors = {}

    for atoms in traj:
        dft_f  = atoms.get_forces()
        atoms.calc = calc
        mlip_f = atoms.get_forces()
        atoms.calc = None

        # find nearest-neighbour distance for each atom
        i_idx, _, dist = neighbor_list('ijd', atoms, cutoff)
        nn_dist = np.full(len(atoms), np.inf)
        for i, d in zip(i_idx, dist):
            nn_dist[i] = min(nn_dist[i], d)

        # accumulate squared force error per distance bin
        err2 = np.sum((dft_f - mlip_f) ** 2, axis=1)
        for nd, e2 in zip(nn_dist, err2):
            if nd < np.inf:
                b = int(nd // bin_width) * bin_width
                bin_errors.setdefault(b, []).append(e2)

    del calc; gc.collect(); torch.cuda.empty_cache()

    centers, rmse_vals = [], []
    for b in sorted(bin_errors):
        centers.append(b + bin_width / 2)
        rmse_vals.append(float(np.sqrt(np.mean(bin_errors[b]))))
    return {'bin_centers': centers, 'rmse': rmse_vals}


bin_data = {}
for label, ckpt in {'scratch': scratch_ckpt, **ft_checkpoints}.items():
    print(f'  {label} …')
    bin_data[label] = force_rmse_by_distance(ckpt, TEST_XYZ)

plot_force_rmse_by_bin(
    bin_data    = bin_data,
    output_path = f'{RESULTS_DIR}/force_rmse_by_bin.png',
)

### 4c. Violin plots — Force MAE and Force Softening

Per-structure force MAE and force-softening slope are computed for each model.
Force softening (slope < 1 in a DFT-vs-MLIP linear fit) indicates the model
under-predicts large forces — a common issue when the training set does not
cover extreme configurations.

In [ ]:
from sklearn.linear_model import LinearRegression
from sevenn.calculator import SevenNetCalculator

def per_structure_metrics(checkpoint, test_xyz):
    traj = ase_read(test_xyz, index=':')
    calc = SevenNetCalculator(checkpoint)
    mae_vals, slope_vals = [], []

    for atoms in traj:
        dft_f  = atoms.get_forces().flatten()
        atoms.calc = calc
        mlip_f = atoms.get_forces().flatten()
        atoms.calc = None

        mae_vals.append(float(np.mean(np.abs(dft_f - mlip_f))))

        # linear fit: mlip_f = slope * dft_f; slope < 1 means force softening
        valid = ~(np.isnan(dft_f) | np.isnan(mlip_f))
        if valid.sum() >= 2:
            reg = LinearRegression().fit(dft_f[valid].reshape(-1, 1), mlip_f[valid])
            slope_vals.append(float(reg.coef_[0]))

    del calc; gc.collect(); torch.cuda.empty_cache()
    return {'mae': np.array(mae_vals), 'softening': np.array(slope_vals)}


# compute per-structure metrics for the scratch model once, reuse across methods
scratch_metrics = per_structure_metrics(scratch_ckpt, TEST_XYZ)
violin_mae, violin_soft = {}, {}

for method in ['random', 'direct', 'lcmd']:
    ckpt = ft_checkpoints.get(method)
    if ckpt is None:
        continue
    ft_m = per_structure_metrics(ckpt, TEST_XYZ)
    # split violin: left half = scratch, right half = fine-tuned
    violin_mae[method]  = {'scratch': scratch_metrics['mae'],      'ft': ft_m['mae']}
    violin_soft[method] = {'scratch': scratch_metrics['softening'], 'ft': ft_m['softening']}

plot_violin(
    violin_mae,
    output_path    = f'{RESULTS_DIR}/violin_force_mae.png',
    metric         = 'Force MAE (eV/Å)',
    ylim           = (0, 0.3),
)
# reference line at 1.0 — perfect slope (no softening)
plot_violin(
    violin_soft,
    output_path    = f'{RESULTS_DIR}/violin_force_softening.png',
    metric         = 'Force softening slope',
    ylim           = (0.90, 1.05),
    reference_line = 1.0,
)

In [ ]:
print('All done.')
print(f'Models  →  {TRAINING_DIR}/')
print(f'Figures →  {RESULTS_DIR}/')